<a href="https://colab.research.google.com/github/Prad0510/ATM-Machine/blob/main/BinaryClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [14]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 18 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         750000 non-null  int64 
 1   age        750000 non-null  int64 
 2   job        750000 non-null  object
 3   marital    750000 non-null  object
 4   education  750000 non-null  object
 5   default    750000 non-null  object
 6   balance    750000 non-null  int64 
 7   housing    750000 non-null  object
 8   loan       750000 non-null  object
 9   contact    750000 non-null  object
 10  day        750000 non-null  int64 
 11  month      750000 non-null  object
 12  duration   750000 non-null  int64 
 13  campaign   750000 non-null  int64 
 14  pdays      750000 non-null  int64 
 15  previous   750000 non-null  int64 
 16  poutcome   750000 non-null  object
 17  y          750000 non-null  int64 
dtypes: int64(9), object(9)
memory usage: 103.0+ MB


In [15]:
X = train.drop(columns=['y'])
y = train['y']

In [16]:
X = X.drop(columns=['id'])
X_test = test.drop(columns=['id'])

In [17]:
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

print("Categorical:", len(cat_cols))
print("Numerical:", len(num_cols))

Categorical: 9
Numerical: 7


In [18]:
X_full = pd.concat([X, X_test], axis=0)
X_full_encoded = pd.get_dummies(
    X_full,columns=cat_cols,drop_first=True)
X_train_encoded = X_full_encoded[:len(X)]
X_test_encoded = X_full_encoded[len(X):]

In [19]:
print(X_train_encoded.shape, X_test_encoded.shape)

(750000, 42) (250000, 42)


In [20]:
from sklearn.model_selection import train_test_split

nan_indices = y[y.isna()].index
y_cleaned = y.drop(nan_indices)
X_train_encoded_cleaned = X_train_encoded.drop(nan_indices)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_encoded_cleaned,
    y_cleaned,
    test_size=0.2,
    stratify=y_cleaned,
    random_state=42
)


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

lr = LogisticRegression(max_iter=1000, n_jobs=-1)

lr.fit(X_tr, y_tr)

val_probs = lr.predict_proba(X_val)[:,1]

roc_auc = roc_auc_score(y_val, val_probs)
print("LR ROC-AUC:", roc_auc)

LR ROC-AUC: 0.9396312290100267


In [22]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

gbr = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=30,
    random_state=42
)

gbr.fit(X_tr, y_tr)

val_probs = gbr.predict_proba(X_val)[:,1]
roc_auc = roc_auc_score(y_val, val_probs)

print("GB ROC-AUC:", roc_auc)


GB ROC-AUC: 0.9552923878665186


In [23]:
train_probs = gbr.predict_proba(X_tr)[:,1]

from sklearn.metrics import roc_auc_score

print("Train ROC-AUC:", roc_auc_score(y_tr, train_probs))
print("Val ROC-AUC:", roc_auc_score(y_val, val_probs))


Train ROC-AUC: 0.9545863862232585
Val ROC-AUC: 0.9552923878665186


In [26]:
final_gbr = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=30,
    random_state=42
)

final_gbr.fit(X_train_encoded, y)


GradientBoostingClassifier(learning_rate=0.05, min_samples_leaf=30,
                           n_estimators=200, random_state=42)

In [28]:
test_probs = final_gbr.predict_proba(X_test_encoded)[:,1]


In [30]:
test_ids = test['id']
submission = pd.DataFrame({
    'id': test_ids,
    'y': test_probs
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,y
0,750000,0.031555
1,750001,0.157163
2,750002,0.006053
3,750003,0.003480
4,750004,0.044858


In [31]:
print(submission.dtypes)
print(submission.isnull().sum())

id      int64
y     float64
dtype: object
id    0
y     0
dtype: int64
